In [1]:
import pandas as pd
import numpy as np

In [2]:
raw = pd.read_excel("raw/CKADTV0-CKIDTVO marzo 2026.xls", sheet_name='CKADTV0').to_dict('records')
output_as = "json/marzo_2026.json"
print(len(raw))

15919


In [3]:
filtered = [ r for r in raw if r['QUOTA CARBURANTE'] and not np.isnan(r['QUOTA CARBURANTE'])]
print(len(filtered))

15909


In [4]:
factors = {
    "petrol": 1,
    "carDiesel": 1,
    "methane": 1.0/1.49,
    "gpl": 1,
    "electricity": 1
}

In [5]:
easy_names = {
    "Benzina Verde": "petrol",
    "Ibrido-Benzina": "petrol",
    "Gasolio": "carDiesel",
    "Ibrido-Gasolio": "carDiesel",
    "Gas Propano Liquido": "gpl",
    "Miscela": "carDiesel",
    "Metano Esclusivo": "methane",
    "Elettrica": "electricity",
}

easy_ones = { k: [] for k in easy_names.keys() }

for f in filtered:
    if( f['ALIMENTAZIONE'] in easy_ones ):
        f['consumption'] = {
            easy_names[f['ALIMENTAZIONE']]: float(f['CONSUMO CARBURANTE']) * factors[easy_names[f['ALIMENTAZIONE']]]
        }
        easy_ones[f['ALIMENTAZIONE']].append(f)

for k in easy_ones.keys():
    print(k, len(easy_ones[k]))

Benzina Verde 7746
Ibrido-Benzina 722
Gasolio 5650
Ibrido-Gasolio 313
Gas Propano Liquido 1
Miscela 11
Metano Esclusivo 41
Elettrica 497


In [6]:
hybrid_names = {
    "Benzina e Gas Liquido": ["petrol","gpl"],
    "Benzina e Metano": ["petrol","methane"],
    "Ibrido-Benzina-Gpl": ["petrol","gpl"],
}

hybrid = { k: [] for k in hybrid_names.keys() }

for f in filtered:
    if( f['ALIMENTAZIONE'] in hybrid_names ):
        f['consumption'] = {
            hybrid_names[f['ALIMENTAZIONE']][0]: float(f['CONSUMO CARBURANTE']) * 0.1 * factors[hybrid_names[f['ALIMENTAZIONE']][0]],
            hybrid_names[f['ALIMENTAZIONE']][1]: float(f['CONSUMO CARBURANTE']) * 0.9 * factors[hybrid_names[f['ALIMENTAZIONE']][1]],
        }
        hybrid[f['ALIMENTAZIONE']].append(f)

for k in hybrid.keys():
    print(k, len(hybrid[k]))

Benzina e Gas Liquido 469
Benzina e Metano 96
Ibrido-Benzina-Gpl 7


In [8]:
plugin_names = {
    "Ibrido plugin-Benzina": "petrol",
    "Ibrido plugin-Gasolio": "carDiesel",
}

plugin_prices = {
    "petrol": 1.67,
    "carDiesel": 1.721,
    "electricity": 0.484
}

plugin = { k: [] for k in plugin_names.keys() }

for f in filtered:
    if( f['ALIMENTAZIONE'] in plugin_names ):
        f['consumption'] = {
            plugin_names[f['ALIMENTAZIONE']]: float(f['CONSUMO CARBURANTE']) / 2.0 * factors[plugin_names[f['ALIMENTAZIONE']]]
        }
        quota_solo_carb = f['consumption'][plugin_names[f['ALIMENTAZIONE']]] * plugin_prices[plugin_names[f['ALIMENTAZIONE']]] / 100.0
        quota_solo_elettrica = float(f['QUOTA CARBURANTE']) - quota_solo_carb
        f['consumption']['electricity'] = quota_solo_elettrica / plugin_prices['electricity'] * 100
        plugin[f['ALIMENTAZIONE']].append(f)

for k in plugin.keys():
    print(k, len(plugin[k]))

Ibrido plugin-Benzina 338
Ibrido plugin-Gasolio 18


In [9]:
for f in filtered:
    if( 'consumption' not in f ):
        print(f)

In [10]:
json_ready = [
    {
        'power': f['ALIMENTAZIONE'],
        'label': f['MARCA'] + " " + f['MODELLO'],
        'QUOTA_CAPITALE': f['QUOTA CAPITALE'],
        'consumption': f['consumption'],
        'priceperkm_estimated': f['QUOTA CAPITALE'] + f['QUOTA CARBURANTE'] + f['QUOTA PNEUMATICI'] + f['QUOTA MANUTENZIONE'],
        'QUOTA_PNEUMATICI': f['QUOTA PNEUMATICI'],
        'QUOTA_MANUTENZIONE': f['QUOTA MANUTENZIONE'],
    } for f in filtered
]

In [11]:
import json
with open(output_as, 'w') as f:
    json.dump(json_ready, f)